# Sonata View Inspection

This notebook inspects the input pipeline for the Sonata self-distillation recipe.

It answers:

- what does a single event look like after the Sonata view transform?
- do global and local views have the expected point counts and spatial extent?
- is the energy transform (`log`) applied correctly?
- how does masking inside the model partition the global view into patches?
- do matched neighbours between student and teacher views land in the right places?
- what does a full `SonataBatch` look like before it enters the model?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from collider_fm.data import ColliderMLDataset
from collider_fm.project_config import load_project_config
from collider_fm.views import (
    build_point_view_from_event,
    build_sonata_batch,
    augment_point_view,
    transform_total_energy,
    normalize_coord,
    POINT_FEATURE_DIM,
)
from collider_fm.sonata_model import SonataSelfDistillation
from collider_fm.model import create_training_sonata_model

plt.style.use('default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = False

## Configuration

In [ ]:
SEED = 42
CONFIG = load_project_config()
DATA_CONFIG = CONFIG.data
SV = CONFIG.sonata_views
SM = CONFIG.model.sonata_training

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Device:', DEVICE)
print('Energy transform:', SV.energy_transform)
print('Coord center:', SV.coord_center)
print('Coord scale:', SV.coord_scale)
print('Global crop range:', SV.global_crop_min_ratio, '-', SV.global_crop_max_ratio)
print('Local crop range:', SV.local_crop_min_ratio, '-', SV.local_crop_max_ratio)
print('Num global views:', SV.num_global_views)
print('Num local views:', SV.num_local_views)

## Load A Few Real Events

In [ ]:
ds = ColliderMLDataset(
    dataset_name=DATA_CONFIG.dataset_name,
    split='train[:5]',
    dataset_type=DATA_CONFIG.dataset_type,
    pu_config=DATA_CONFIG.pu_config,
    cache_dir=DATA_CONFIG.cache_dir,
    dataset_revision=DATA_CONFIG.dataset_revision,
    local_files_only=True,
    object_types=['calo_hits'],
)
events = [ds[i] for i in range(len(ds))]
for i, ev in enumerate(events):
    n = len(ev['calo_hits']['x'])
    e = ev['calo_hits']['total_energy']
    print(f'Event {i}: {n} hits, energy [{e.min():.2f}, {e.max():.2f}]')

## Base Point View: Raw vs Energy-Transformed

Before building multi-view batches, inspect what `build_point_view_from_event` does to a single event.

In [ ]:
event = events[0]

base_view = build_point_view_from_event(
    event,
    device=DEVICE,
    grid_size=float(SM.grid_size),
    coord_center=SV.coord_center,
    coord_scale=SV.coord_scale,
    energy_transform=SV.energy_transform,
    energy_min=SV.energy_min,
    energy_max=SV.energy_max,
)

coord = base_view['coord']
feat = base_view['feat']
energy_feat = feat[:, 3]

print(f'coord shape: {tuple(coord.shape)}')
print(f'feat shape:  {tuple(feat.shape)}')
print(f'coord range: x [{coord[:,0].min():.1f}, {coord[:,0].max():.1f}]'
      f'  y [{coord[:,1].min():.1f}, {coord[:,1].max():.1f}]'
      f'  z [{coord[:,2].min():.1f}, {coord[:,2].max():.1f}]')
print(f'energy feat range: [{energy_feat.min():.3f}, {energy_feat.max():.3f}]')
print(f'energy transform:  {SV.energy_transform}')
print(f'coord == origin_coord: {torch.equal(coord, base_view["origin_coord"])}')

In [ ]:
raw_energy = event['calo_hits']['total_energy'].float()
transformed = transform_total_energy(
    raw_energy, transform=SV.energy_transform,
    min_val=SV.energy_min, max_val=SV.energy_max,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(raw_energy.numpy(), bins=60, color='tab:blue', alpha=0.85)
axes[0].set_title('Raw calo-hit energy')
axes[0].set_xlabel('energy')
axes[0].set_ylabel('count')
axes[1].hist(transformed.numpy(), bins=60, color='tab:orange', alpha=0.85)
axes[1].set_title(f'Energy after "{SV.energy_transform}" transform')
axes[1].set_xlabel('transformed energy')
axes[1].set_ylabel('count')
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
c = energy_feat.cpu().numpy()
xyz = coord.cpu().numpy()
sc = ax.scatter(xyz[:, 2], xyz[:, 0], xyz[:, 1], c=c, cmap='inferno', s=3, alpha=0.5)
fig.colorbar(sc, ax=ax, shrink=0.6, label='energy (transformed)')
ax.set_xlabel('z')
ax.set_ylabel('x')
ax.set_zlabel('y')
ax.set_title('Base view geometry (one event)')
plt.tight_layout()
plt.show()

## Individual Augmented Views

Sonata builds global views (large crops, used by teacher and masked student) and local views (small crops, student only). Inspect a few independently before batching.

In [ ]:
global_views = []
local_views = []

for i in range(SV.num_global_views):
    from collider_fm.views import _sample_crop_keep_ratio
    keep = _sample_crop_keep_ratio(
        float(SV.global_crop_min_ratio),
        float(SV.global_crop_max_ratio),
        base_view['coord'].device,
    )
    gv = augment_point_view(
        base_view,
        coord_noise_scale=float(SV.coord_noise_scale),
        feat_noise_scale=float(SV.energy_jitter_scale),
        crop_keep_ratio=keep,
        point_dropout=float(SV.point_dropout),
        mask_fraction=0.0,
        view_kind=f'global_{i}',
    )
    global_views.append(gv)
    print(f'Global view {i}: keep={keep:.2f}, points={gv["coord"].shape[0]}')

for i in range(SV.num_local_views):
    keep = _sample_crop_keep_ratio(
        float(SV.local_crop_min_ratio),
        float(SV.local_crop_max_ratio),
        base_view['coord'].device,
    )
    lv = augment_point_view(
        base_view,
        coord_noise_scale=float(SV.coord_noise_scale),
        feat_noise_scale=float(SV.energy_jitter_scale),
        crop_keep_ratio=keep,
        point_dropout=float(SV.point_dropout),
        mask_fraction=0.0,
        view_kind=f'local_{i}',
    )
    local_views.append(lv)
    print(f'Local view {i}:  keep={keep:.2f}, points={lv["coord"].shape[0]}')

In [ ]:
n_global = len(global_views)
n_local = len(local_views)
n_cols = n_global + n_local

fig = plt.figure(figsize=(4 * n_cols, 5))

for i, gv in enumerate(global_views):
    ax = fig.add_subplot(1, n_cols, i + 1, projection='3d')
    xyz = gv['coord'].cpu().numpy()
    e = gv['feat'][:, 3].cpu().numpy()
    ax.scatter(xyz[:, 2], xyz[:, 0], xyz[:, 1], c=e, cmap='inferno', s=3, alpha=0.5)
    ax.set_title(f'Global {i}\n({xyz.shape[0]} pts)', fontsize=9)
    ax.set_xlabel('z', fontsize=7)

for i, lv in enumerate(local_views):
    ax = fig.add_subplot(1, n_cols, n_global + i + 1, projection='3d')
    xyz = lv['coord'].cpu().numpy()
    e = lv['feat'][:, 3].cpu().numpy()
    ax.scatter(xyz[:, 2], xyz[:, 0], xyz[:, 1], c=e, cmap='inferno', s=3, alpha=0.5)
    ax.set_title(f'Local {i}\n({xyz.shape[0]} pts)', fontsize=9)
    ax.set_xlabel('z', fontsize=7)

plt.suptitle('Sonata views for one event', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
gv0 = global_views[0]
fig = plt.figure(figsize=(12, 5))

ax0 = fig.add_subplot(1, 2, 1, projection='3d')
orig = gv0['origin_coord'].cpu().numpy()
e = gv0['feat'][:, 3].cpu().numpy()
ax0.scatter(orig[:, 2], orig[:, 0], orig[:, 1], c=e, cmap='inferno', s=3, alpha=0.5)
ax0.set_title('origin_coord (pre-rotation)')
ax0.set_xlabel('z')

ax1 = fig.add_subplot(1, 2, 2, projection='3d')
aug = gv0['coord'].cpu().numpy()
ax1.scatter(aug[:, 2], aug[:, 0], aug[:, 1], c=e, cmap='inferno', s=3, alpha=0.5)
ax1.set_title('coord (after rotation + jitter)')
ax1.set_xlabel('z')

plt.suptitle('Global view 0: origin_coord vs coord', fontsize=11)
plt.tight_layout()
plt.show()

diff = (gv0['coord'] - gv0['origin_coord']).norm(dim=-1)
print(f'coord vs origin_coord diff: mean={diff.mean():.2f}, max={diff.max():.2f} mm')

## Full SonataBatch

Build a `SonataBatch` from multiple events and inspect the packed tensors.

In [ ]:
batch = build_sonata_batch(
    events,
    device=DEVICE,
    max_calo_hits=SV.max_calo_hits,
    grid_size=float(SM.grid_size),
    coord_noise_scale=float(SV.coord_noise_scale),
    feat_noise_scale=float(SV.energy_jitter_scale),
    point_dropout=float(SV.point_dropout),
    num_global_views=int(SV.num_global_views),
    num_local_views=int(SV.num_local_views),
    global_crop_min_ratio=float(SV.global_crop_min_ratio),
    global_crop_max_ratio=float(SV.global_crop_max_ratio),
    local_crop_min_ratio=float(SV.local_crop_min_ratio),
    local_crop_max_ratio=float(SV.local_crop_max_ratio),
    coord_center=SV.coord_center,
    coord_scale=SV.coord_scale,
    energy_transform=SV.energy_transform,
    energy_min=SV.energy_min,
    energy_max=SV.energy_max,
)

for key, val in batch.items():
    if isinstance(val, torch.Tensor):
        print(f'{key:>25s}: shape={str(tuple(val.shape)):>20s}  dtype={val.dtype}')

In [ ]:
n_events = len(events)
g_off = batch['global_offset']
l_off = batch['local_offset']

g_counts = torch.diff(g_off, prepend=g_off.new_zeros(1))
l_counts = torch.diff(l_off, prepend=l_off.new_zeros(1))

print(f'Events in batch: {n_events}')
print(f'Global views total: {g_off[-1].item()} points across {g_off.numel()} views')
print(f'Local  views total: {l_off[-1].item()} points across {l_off.numel()} views')
print(f'Global view point counts: {g_counts.tolist()}')
print(f'Local  view point counts: {l_counts.tolist()}')
print(f'Global coord range: x=[{batch["global_coord"][:,0].min():.1f}, {batch["global_coord"][:,0].max():.1f}]'
      f'  y=[{batch["global_coord"][:,1].min():.1f}, {batch["global_coord"][:,1].max():.1f}]'
      f'  z=[{batch["global_coord"][:,2].min():.1f}, {batch["global_coord"][:,2].max():.1f}]')
print(f'Local  coord range: x=[{batch["local_coord"][:,0].min():.1f}, {batch["local_coord"][:,0].max():.1f}]'
      f'  y=[{batch["local_coord"][:,1].min():.1f}, {batch["local_coord"][:,1].max():.1f}]'
      f'  z=[{batch["local_coord"][:,2].min():.1f}, {batch["local_coord"][:,2].max():.1f}]')

g_feat = batch['global_feat']
print(f'Global feat finite: {torch.isfinite(g_feat).all().item()}')
print(f'Global energy feat range: [{g_feat[:,3].min():.3f}, {g_feat[:,3].max():.3f}]')

## Model-Level Masking

Sonata generates masks inside `forward()` from the global view coordinates.
Inspect what `generate_mask` produces with the current `mask_size` and `mask_ratio`.

In [ ]:
model = create_training_sonata_model(device=DEVICE)
model.eval()
print(f'Model params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')
print(f'mask_size: {model.mask_size}')
print(f'mask_ratio: {model.mask_ratio}')
print(f'match_max_r: {model.match_max_r}')
print(f'mask_jitter: {model.mask_jitter}')

In [ ]:
with torch.no_grad():
    mask, point_cluster = model.generate_mask(
        batch['global_coord'], batch['global_offset']
    )

n_patches = point_cluster.unique().numel()
n_masked = mask.sum().item()
n_total = mask.numel()
print(f'Number of patches (mask_size={model.mask_size:.0f}mm): {n_patches}')
print(f'Masked points: {n_masked}/{n_total} ({n_masked/n_total:.1%})')
print(f'Configured mask_ratio: {model.mask_ratio:.1%}')

cluster_sizes = torch.bincount(point_cluster)
print(f'Points per patch: min={cluster_sizes.min().item()}, '
      f'median={cluster_sizes.float().median().item():.0f}, '
      f'max={cluster_sizes.max().item()}')

In [ ]:
g_coord = batch['global_coord'].cpu().numpy()
mask_np = mask.cpu().numpy()

n_view_events = batch['global_offset'].numel()
plot_views = min(2, n_view_events)

fig = plt.figure(figsize=(7 * plot_views, 5))
starts = [0] + batch['global_offset'][:-1].tolist()
ends = batch['global_offset'].tolist()

for vi in range(plot_views):
    ax = fig.add_subplot(1, plot_views, vi + 1, projection='3d')
    s, e = starts[vi], ends[vi]
    view_coord = g_coord[s:e]
    view_mask = mask_np[s:e]
    unmasked = ~view_mask
    ax.scatter(view_coord[unmasked, 2], view_coord[unmasked, 0], view_coord[unmasked, 1],
               c='tab:blue', s=2, alpha=0.3, label='visible')
    ax.scatter(view_coord[view_mask, 2], view_coord[view_mask, 0], view_coord[view_mask, 1],
               c='tab:red', s=2, alpha=0.3, label='masked')
    ax.set_title(f'Global view {vi}\nmasked={view_mask.sum()}/{len(view_mask)}', fontsize=9)
    ax.set_xlabel('z')
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle(f'Masked vs visible points (mask_size={model.mask_size:.0f}mm)', fontsize=11)
plt.tight_layout()
plt.show()

## Neighbour Matching

`match_neighbour` finds corresponding points between the masked student global view and the teacher global view using `origin_coord`.
This is the alignment that makes the distillation loss work.

In [ ]:
with torch.no_grad():
    match_idx = model.match_neighbour(
        batch['global_origin_coord'], batch['global_offset'],
        batch['global_origin_coord'], batch['global_offset'],
    )

print(f'Matched pairs: {match_idx.shape[0]}')
print(f'match_max_r: {model.match_max_r:.1f}mm')

if match_idx.shape[0] > 0:
    src = batch['global_origin_coord'][match_idx[:, 0]]
    tgt = batch['global_origin_coord'][match_idx[:, 1]]
    distances = (src - tgt).norm(dim=-1)
    print(f'Match distances: mean={distances.mean():.2f}, max={distances.max():.2f}mm')

    g_off = batch['global_offset']
    n_view_events = g_off.numel()
    batch_idx = torch.arange(n_view_events, device=g_off.device).repeat_interleave(
        torch.diff(g_off, prepend=g_off.new_zeros(1))
    )
    matches_per_view = torch.bincount(batch_idx[match_idx[:, 0]], minlength=n_view_events)
    points_per_view = torch.diff(g_off, prepend=g_off.new_zeros(1))
    match_frac = matches_per_view.float() / points_per_view.float().clamp_min(1)
    print(f'Match fraction per view: {match_frac.tolist()}')

## Forward Pass Summary

Run one forward pass and inspect the loss breakdown.

In [ ]:
model.setup_schedulers(total_steps=100)

with torch.no_grad():
    result = model(batch)

for key, val in result.items():
    if isinstance(val, torch.Tensor) and val.numel() == 1:
        print(f'{key:>25s}: {val.item():.4f}')
    elif isinstance(val, torch.Tensor):
        print(f'{key:>25s}: shape={tuple(val.shape)}')

monitor = model.last_monitoring_state
print(f'\nmasked_fraction: {monitor["masked_fraction"]:.3f}')
if monitor['student_logits'] is not None:
    print(f'student_logits shape: {tuple(monitor["student_logits"].shape)}')
if monitor['point_features'] is not None:
    print(f'point_features shape: {tuple(monitor["point_features"].shape)}')

## Sanity Checks

A few quick checks to confirm the pipeline is healthy:

- all feature tensors finite
- global views contain more points than local views
- mask fraction is in the expected range
- matched pairs exist (loss is not trivially zero)
- energy transform produces a reasonable range

In [ ]:
checks = []

feat_finite = torch.isfinite(batch['global_feat']).all() and torch.isfinite(batch['local_feat']).all()
checks.append(('All features finite', feat_finite.item()))

global_pts = batch['global_coord'].shape[0]
local_pts = batch['local_coord'].shape[0]
checks.append(('Global > local point count', global_pts > local_pts))

g_counts = torch.diff(batch['global_offset'], prepend=batch['global_offset'].new_zeros(1))
l_counts = torch.diff(batch['local_offset'], prepend=batch['local_offset'].new_zeros(1))
checks.append(('Global views > local views per event', (g_counts.mean() > l_counts.mean()).item()))

checks.append(('Mask fraction > 0', monitor['masked_fraction'] > 0))
checks.append(('Mask fraction < 1', monitor['masked_fraction'] < 1))
checks.append(('Loss is finite', torch.isfinite(result['loss']).item()))
checks.append(('Loss > 0', (result['loss'] > 0).item()))
checks.append(('Match pairs found', match_idx.shape[0] > 0))

e_range_ok = batch['global_feat'][:, 3].abs().max() < 100
checks.append(('Energy feat range reasonable', e_range_ok.item()))

print('Sanity checks:')
for label, ok in checks:
    status = 'PASS' if ok else 'FAIL'
    print(f'  [{status}] {label}')